# ESCM 2.0 — why does a full edge fail to persist?

Notebook 08 demonstrated that baseline and additive inhibitory-current models produce **identical output spike trains** for all 12 moving-edge configurations (six protocols at two neighborhood radii). However, identical outputs do not mean the output is desirable:

- Simultaneous stimulation produces brief firing and **no sustained edge**.
- Staggered stimulation allows **only the final activated neuron** to continue firing, rather than preserving the entire five-neuron edge.

**Research question.** Is the disappearance of the complete edge driven by the relative timing of the five inputs, by inhibitory strength, or by the neighborhood radius?

We now isolate these factors using the **counter-free candidate**. All other neuronal rules stay fixed: same persistence gate, self-excitation, input weight, membrane equation, timestep, and complete square neighborhood.

This is a *parameter-sensitivity experiment*, not a claim that reducing inhibition is necessarily biologically preferable. We measure the representation's spatial coverage, not only its total spike count.

## 1. Define the input and outcome

A five-pixel vertical line moves through columns 2–6 of a 9 × 7 grid, with 4 ms between columns. Within each column the five camera events may be simultaneous or staggered by 0.5, 1 or 2 ms per row.

By 45 ms, all these input protocols have ended. We evaluate activity in **45–60 ms**. For each simulation we measure:

\[
\text{final-edge coverage}=\#\{i\text{ in final 5 pixels}:i\text{ spikes during }[45,60)\},
\]

and **old-edge coverage**, defined analogously for the 20 pixels of previous columns. The ideal moving-edge outcome has final-edge coverage close to five and old-edge coverage close to zero.

We also retain counts of firing outside any stimulated edge, so the model cannot appear successful merely by producing indiscriminate activity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from brian2 import *

prefs.codegen.target = "numpy"

width, height = 9, 7
N = width * height
rows = np.arange(1, 6)
columns = np.arange(2, 7)
last_edge = rows * width + columns[-1]
old_edges = np.concatenate([rows * width + col for col in columns[:-1]])
stimulated = np.concatenate([old_edges, last_edge])

dt = 0.5 * ms
tau = 0.2 * ms
vt, vr, beta, Wi = 0.1, 0.0, 0.5, 6.0
end_ms, evaluation_start_ms = 60.0, 45.0
interval_ms = 4.0

stagger_ms_values = (0.0, 0.5, 1.0, 2.0)
inhibitory_weights = (-3.0, -1.5, -0.75, 0.0)
radii = (2, 8)

def make_stimulus(stagger_ms):
    ids = np.array([row*width+col for col in columns for row in rows], dtype=int)
    times = np.array([1.0 + step*interval_ms + r*stagger_ms
                      for step in range(len(columns)) for r in range(len(rows))])
    assert times.max() < evaluation_start_ms
    return ids, times

print("Final edge neuron indices:", last_edge)
print(f"Experiment: {len(stagger_ms_values)*len(inhibitory_weights)*len(radii)} runs")

## 2. Run the counter-free model

The only parameter varied across runs is the inhibitory weight \(W_k\), along with the **input timing** and neighborhood radius. We retain:

\[
\tau\dot v_i=-v_i+\mathrm{ExtIn}_i+P_i(\mathrm{Exc}_i+\mathrm{Inh}_i),
\]

with \(\mathrm{Inh}_i\leftarrow\mathrm{Inh}_i+P_iW_k\) whenever a neighbor fires.

The \(W_k=0\) condition is a control without inhibition: it should preserve input-driven activity but may fail to suppress older positions. We intentionally do not tune excitation, threshold, \(P\), refractory period, or decay time constants in this experiment.

In [ ]:
def simulate(stagger_ms, inhibitory_weight, radius):
    start_scope()
    defaultclock.dt = dt
    params = dict(tau_model=tau, threshold_model=vt, reset_model=vr,
                  beta_model=beta, self_weight=Wi, inh_weight=inhibitory_weight)

    ids, times = make_stimulus(stagger_ms)
    camera = SpikeGeneratorGroup(N, ids, times*ms)
    equations = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau_model : 1
    dExtIn/dt = -ExtIn/tau_model : 1
    dExc/dt = -Exc/tau_model : 1
    dInh/dt = -Inh/tau_model : 1
    P : integer
    """
    neurons = NeuronGroup(
        N, equations, threshold="v > threshold_model",
        reset="v = reset_model", refractory=0*ms,
        events={"P_ON": "v > threshold_model",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= threshold_model)"},
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta_model", namespace=params)
    feed.connect(j="i")
    recurrent = Synapses(
        neurons, neurons,
        on_pre={"pre": "Exc_pre = self_weight",
                "pre_2": "Inh_post += P_post * inh_weight"},
        namespace=params,
    )
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1
    src = np.repeat(np.arange(N), N)
    dst = np.tile(np.arange(N), N)
    adjacent = ((src != dst)
                & (abs(src % width - dst % width) <= radius)
                & (abs(src // width - dst // width) <= radius))
    recurrent.connect(i=src[adjacent], j=dst[adjacent])

    output = SpikeMonitor(neurons)
    net = Network(camera, neurons, feed, recurrent, output)
    net.run(end_ms*ms)
    return np.asarray(output.t/ms), np.asarray(output.i, dtype=int)

results = []
trains = {}
for radius in tqdm(radii, desc="Neighborhood radius"):
    for stagger in stagger_ms_values:
        for weight in inhibitory_weights:
            times, ids = simulate(stagger, weight, radius)
            trains[radius, stagger, weight] = (times, ids)
            late = ids[(times >= evaluation_start_ms) & (times < end_ms)]
            active = np.unique(late)
            results.append({
                "k": radius, "stagger_ms": stagger, "Wk": weight,
                "final_pixels_active": np.isin(last_edge, active).sum(),
                "old_pixels_active": np.isin(old_edges, active).sum(),
                "other_pixels_active": np.count_nonzero(~np.isin(active, stimulated)),
                "final_spikes": np.isin(late, last_edge).sum(),
                "old_spikes": np.isin(late, old_edges).sum(),
                "total_spikes": len(ids),
            })
summary = pd.DataFrame(results)
print(summary.to_string(index=False))

## 3. Interpret spatial retention, not total activity

The heatmaps show how many of the five pixels making up the **last edge** continue firing in the final 15 ms. This avoids counting spikes immediately after the last input as persistence.

In addition, the printed table shows old-edge activity. A condition with all five final pixels active is not desirable if it also retains all twenty previous pixels. We want **selective** retention.

A favorable result is a parameter region, not a single lucky configuration, with high final-edge coverage and low old-edge coverage.

In [ ]:
fig, axes = plt.subplots(1, len(radii), figsize=(12, 4.5), constrained_layout=True)
for ax, radius in zip(axes, radii):
    subset = summary[summary.k == radius]
    matrix = subset.pivot(index="Wk", columns="stagger_ms", values="final_pixels_active")
    matrix = matrix.loc[list(inhibitory_weights), list(stagger_ms_values)]
    im = ax.imshow(matrix.values, cmap="viridis", vmin=0, vmax=5, aspect="auto")
    ax.set_xticks(range(len(stagger_ms_values)), labels=stagger_ms_values)
    ax.set_yticks(range(len(inhibitory_weights)), labels=inhibitory_weights)
    ax.set(xlabel="Stagger per row (ms)", ylabel="Inhibitory weight Wk",
           title=f"Final-edge coverage (k={radius})")
    for (i, j), value in np.ndenumerate(matrix.values):
        ax.text(j, i, str(int(value)), ha="center", va="center",
                color="white" if value < 3 else "black")
fig.colorbar(im, ax=axes, label="Final-edge neurons active (of 5)")
plt.show()

print("\nConfigurations sorted by final coverage, then fewest old pixels:")
display(summary.sort_values(
    ["final_pixels_active", "old_pixels_active", "other_pixels_active"],
    ascending=[False, True, True]).head(12))

## 4. Inspect two contrasting spike trains

Compare the original candidate setting \(k=2, W_k=-3,\) stagger 0.5 ms to a setting with the strongest observed selectivity. The comparison is chosen from the computed measurements; **it does not establish generalization**.

Each raster shows output spikes across all 63 neurons, with input events marked as red crosses. The vertical line marks the last input timestamp.

In [ ]:
baseline_choice = (2, 0.5, -3.0)
best = summary.sort_values(
    ["final_pixels_active", "old_pixels_active", "other_pixels_active"],
    ascending=[False, True, True]).iloc[0]
best_choice = (int(best.k), float(best.stagger_ms), float(best.Wk))
choices = [("Original candidate", baseline_choice), ("Best measured selectivity", best_choice)]
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, (label, key) in zip(axes, choices):
    t, ids = trains[key]
    stim_ids, stim_times = make_stimulus(key[1])
    ax.scatter(t, ids, s=12, alpha=0.75, label="Output spikes")
    ax.scatter(stim_times, stim_ids, s=40, marker="x", color="tab:red", label="Input events")
    ax.axvline(stim_times.max(), color="black", linestyle="--", label="Final input")
    ax.axvspan(evaluation_start_ms, end_ms, color="0.9", alpha=0.5)
    ax.set(ylabel="Neuron index", title=f"{label}: k={key[0]}, stagger={key[1]} ms, Wk={key[2]}")
axes[0].legend(ncol=3)
axes[-1].set(xlabel="Time (ms)", xlim=(0, end_ms), ylim=(0, N))
plt.show()

## 5. Scientific conclusion and next step

The previous experiments showed that removing the spike counter can preserve **exact output spikes** over a narrow tested domain. Notebook 08 exposed a separate, more important limitation: **multi-pixel edges are not retained as a complete structure** under the tested legacy parameters.

This experiment asks whether that limitation is attributable to simultaneous activation, inhibitory strength or radius. If no region preserves the five-pixel edge while suppressing earlier locations, the model likely needs a **qualitative redesign of its inhibitory organization**, not simply a different inhibitory weight.

Possible next research directions would then include local rather than uniform inhibition, distance-dependent receptive fields, or alternative activity-dependent gating. Those changes should be tested individually against the original model and event-camera recordings.

**Status:** This notebook is prepared for execution on the user's Brian2 installation. No results are assumed. Parameter combinations explored here are descriptive experiments, not an optimized parameter search.